<a href="https://colab.research.google.com/github/Zebedee2021/pyspt-v2/blob/main/tutorials/ch02_generation/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Ch02 - 信号预处理 | Signal Preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zebedee2021/pyspt-v2/blob/main/tutorials/ch02_generation/02_preprocessing.ipynb)

本教程演示如何使用 `pyspt.preprocessing` 模块去除趋势与噪声，包括 `detrend`（去趋势）与 `medfilt1`（一维中值滤波）。

This tutorial demonstrates how to detrend signals and remove impulsive noise using `detrend` and `medfilt1`.

---


## 环境准备 | Setup

在 Google Colab 中运行时，需要先安装 PySPT：


In [ ]:
# Install pyspt (only needed in Colab or fresh environments)
try:
    import pyspt
except ImportError:
    !pip install -q git+https://github.com/Zebedee2021/pyspt-v2.git
    import pyspt

print(f"PySPT version: {pyspt.__version__}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyspt.preprocessing import detrend, medfilt1

# Plot style
plt.rcParams["figure.figsize"] = (12, 3)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


---
## 1. 去趋势 | Detrend

实际信号常带有缓慢漂移（基线、传感器偏置），会影响峰值检测、频谱等后续分析。`detrend` 用于去除均值或最佳拟合直线。

**API**: `detrend(x, type='linear', bp=0, axis=None, overwrite_data=False)`

- `type='linear'` → 移除最佳拟合直线（MATLAB 默认）
- `type='constant'` → 只移除均值
- `bp` → 断点索引向量，启用分段线性去趋势


In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 1, 200)
trend = 3.0 * t + 1.0
noise = 0.2 * rng.standard_normal(t.size)
signal = np.sin(2 * np.pi * 5 * t) + trend + noise

y_linear  = detrend(signal)              # remove best-fit line
y_const   = detrend(signal, type='constant')  # remove mean only

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t, signal, label='original (with trend)')
axes[0].legend()
axes[1].plot(t, y_linear, label="detrend type='linear'", color='C1')
axes[1].legend()
axes[2].plot(t, y_const, label="detrend type='constant'", color='C2')
axes[2].legend()
for ax in axes:
    ax.set_ylabel('amplitude')
axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()


### 1.1 2D 信号与首非单维轴 | 2-D signals

对于 N-D 输入，`axis=None` 默认沿第一个长度大于 1 的轴运行（与 MATLAB 一致）。也可以显示指定 `axis=…`。


In [ ]:
x2 = np.arange(20.0).reshape(4, 5) + 10.0  # strong row-wise bias
print("before:\n", x2)
print("after detrend (default axis=0):\n", detrend(x2))


---
## 2. 一维中值滤波 | Median Filter (`medfilt1`)

中值滤波保留边缘，对孤立刺峰极敏感，是去除干扰顶峰/峰值噪声的优选。

**API**: `medfilt1(x, n=3, axis=None)`

- `n` 必须为正奇数，默认 3
- 两端以 0 补齐（与 MATLAB `medfilt1` 一致）


### 2.1 去除刺峰 | Spike removal

对载有单个大峰值的信号，选择适当窗口 (n=5) 可以一次性移除而不伤及中间小振幅。


In [ ]:
rng = np.random.default_rng(1)
t = np.linspace(0, 1, 200)
x = np.sin(2 * np.pi * 5 * t) + 0.05 * rng.standard_normal(t.size)
x[100] += 4.0  # single strong spike

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, x, label='noisy + spike', color='C0')
axes[0].scatter([t[100]], [x[100]], color='red', zorder=5, label='spike')
axes[0].legend()
for n, color in [(3, 'C1'), (5, 'C2'), (7, 'C3')]:
    axes[1].plot(t, medfilt1(x, n), label=f'medfilt1 n={n}', color=color, alpha=0.8)
axes[1].legend()
for ax in axes:
    ax.set_ylabel('amplitude')
axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()


### 2.2 2D 信号 | 2-D signals

默认沿第一个非单维轴运行，使用 `scipy.ndimage.median_filter` 避免同时对所有轴中值。


In [ ]:
x2 = np.column_stack([np.arange(6.0), np.arange(6.0)])
x2[3, 1] = 100.0  # spike in column 1
y2 = medfilt1(x2, 3)
print("input :\n", x2)
print("medfilt1(x2, 3):\n", y2)
assert y2[3, 1] == 4.0, "median over [2, 100, 4] should be 4"
print("spike at row=3 col=1 replaced by 4.0 ✓")


---
## 3. 综合例子；c_combined Pipeline

一个常见场景：载有线性趋势 + 高频噪声 + 偶发峰值。先用 `detrend` 拆趋势，再用 `medfilt1` 去峰值。


In [ ]:
rng = np.random.default_rng(2)
t = np.linspace(0, 2, 400)
trend = 1.5 * t + 0.5
clean = np.sin(2 * np.pi * 3 * t) + 0.3 * np.cos(2 * np.pi * 12 * t)
noisy = clean + trend + 0.1 * rng.standard_normal(t.size)
noisy[150] += 3.0
noisy[300] -= 2.5

step1 = detrend(noisy, type='linear')
step2 = medfilt1(step1, 5)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, data, c in zip(
    axes,
    [noisy, step1, step2, clean],
    ['C0', 'C1', 'C2', 'C3'],
):
    ax.plot(t, data, color=c)
axes[0].set_title('raw (trend + noise + spikes)')
axes[1].set_title("after detrend('linear')")
axes[2].set_title('after medfilt1 n=5')
axes[3].set_title('ground truth')
for ax in axes:
    ax.set_ylabel('amplitude')
axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()


---

## MATLAB 对照 | MATLAB Comparison

| PySPT (Python) | MATLAB | 说明 |
|:---|:---|:---|
| `detrend(x)` | `detrend(x)` | 默认去除最佳拟合直线，双方输出一致 |
| `detrend(x, type='constant')` | `detrend(x, 'constant')` | 仅移除均值 |
| `detrend(x, bp=[k1,k2,...])` | `detrend(x, bp)` | 分段线性去趋势，参数语义完全一致 |
| `medfilt1(x, 5)` | `medfilt1(x, 5)` | 窗口必须为奇数，两端零补（与 MATLAB 一致） |

**容差注释：**两个函数都由 [`tests/fixtures/preprocessing/*.npz`](https://github.com/Zebedee2021/pyspt-v2/blob/main/tutorials/ch02_generation/../../tests/fixtures/preprocessing/) 中的 MATLAB 黄金 fixture 对照，默认容差 1e-9。

---

*PySPT v2 · Beijing Institute of Technology · [GitHub](https://github.com/Zebedee2021/pyspt-v2)*
